# Model Comparison — Linear Regression vs Random Forest vs Tuned XGBoost on 2025 Holdout

Notebook 04 selected the final model (6-feature linear regression) using 2024 as the test set.
Notebook 05 confirmed it generalised to the 2025 true holdout.

What was missing: an apples-to-apples comparison of Linear Regression vs Random Forest vs Tuned XGBoost on the same 2025 holdout. This notebook closes that gap.

All three models train on the same 2022–2024 data, predict on the same 2025 holdout, use the same 6 features, and are evaluated by the same function.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
import xgboost as xgb

# Project paths
PROJECT_ROOT = Path.cwd().parent
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

# Plot styling — consistent with prior notebooks
plt.rcParams["figure.dpi"] = 100
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

print("Imports OK")
print(f"Project root: {PROJECT_ROOT}")
print(f"Data path:    {DATA_PROCESSED}")

Imports OK
Project root: /Users/ompatil9819gmail.com/F1-Race-Predictor
Data path:    /Users/ompatil9819gmail.com/F1-Race-Predictor/data/processed


## Evaluation function (identical to notebook 05 for direct comparability)

In [2]:
# Load full 4-season dataset
df = pd.read_csv(DATA_PROCESSED / "features_2022_2025.csv")

# Drop DNFs
df = df.dropna(subset=["Position"]).copy()

# Feature set: same 6 features as the final shipped model
features = [
    "GridPosition", "QualifyingPosition", "QualifyingGapToPole",
    "DriverFormLast3", "TeamFormLast3", "IsStreetCircuit",
]

# Drop rows with missing critical features
df_clean = df.dropna(subset=features).copy()

# Train: 2022-2024, Test: 2025 (same as notebook 05)
train_df = df_clean[df_clean["Year"].isin([2022, 2023, 2024])].copy()
test_df = df_clean[df_clean["Year"] == 2025].copy()
test_df["race_id"] = test_df["Year"].astype(str) + "-R" + test_df["Round"].astype(str)

X_train = train_df[features]
y_train = train_df["Position"]
X_test = test_df[features]
y_test = test_df["Position"]
race_ids_test = test_df["race_id"]

print(f"Train: {len(X_train)} rows ({train_df['Round'].nunique()} unique rounds across {sorted(train_df['Year'].unique())})")
print(f"Test:  {len(X_test)} rows ({test_df['Round'].nunique()} unique rounds in 2025)")
print(f"Features: {len(features)}")

Train: 1296 rows (24 unique rounds across [np.int64(2022), np.int64(2023), np.int64(2024)])
Test:  468 rows (24 unique rounds in 2025)
Features: 6


In [3]:
def evaluate_predictions(y_true, y_pred, race_ids, label="Model"):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    eval_df = pd.DataFrame({
        "race_id": race_ids,
        "y_true": y_true.values if hasattr(y_true, "values") else y_true,
        "y_pred": y_pred,
    })

    eval_df["pred_rank"] = eval_df.groupby("race_id")["y_pred"].rank(method="min")
    eval_df["true_rank"] = eval_df.groupby("race_id")["y_true"].rank(method="min")

    actual_winners = eval_df[eval_df["true_rank"] == 1]
    predicted_winners = eval_df[eval_df["pred_rank"] == 1]
    races_with_winner_match = actual_winners.merge(
        predicted_winners, on="race_id", suffixes=("_actual", "_predicted")
    )
    top1_correct = (races_with_winner_match["y_true_actual"] == races_with_winner_match["y_true_predicted"]).sum()
    n_races = eval_df["race_id"].nunique()
    top1_accuracy = top1_correct / n_races

    top3_pred = eval_df[eval_df["pred_rank"] <= 3]
    top3_hits = 0
    for race_id in eval_df["race_id"].unique():
        actual_winner_pos = eval_df[(eval_df["race_id"] == race_id) & (eval_df["true_rank"] == 1)]["y_true"].iloc[0]
        race_top3_predicted = top3_pred[top3_pred["race_id"] == race_id]
        if actual_winner_pos in race_top3_predicted["y_true"].values:
            top3_hits += 1
    top3_accuracy = top3_hits / n_races

    print(f"\n=== {label} ===")
    print(f"  RMSE:           {rmse:.3f} positions")
    print(f"  MAE:            {mae:.3f} positions")
    print(f"  Top-1 accuracy: {top1_accuracy:.1%}  ({top1_correct}/{n_races} races)")
    print(f"  Top-3 accuracy: {top3_accuracy:.1%}  ({top3_hits}/{n_races} races)")

    return {"rmse": rmse, "mae": mae, "top1": top1_accuracy, "top3": top3_accuracy}

print("evaluate_predictions defined")

evaluate_predictions defined


## Model 1: Pole baseline (reference floor)

The simplest possible "model" — predict that every driver finishes in the same position they qualified. Strong baseline because qualifying is highly predictive of race outcomes, especially in modern F1.

In [4]:
results_pole = evaluate_predictions(
    y_test, test_df["QualifyingPosition"].values, race_ids_test,
    label="Pole baseline (predict qualifying = finish)",
)


=== Pole baseline (predict qualifying = finish) ===
  RMSE:           4.686 positions
  MAE:            3.252 positions
  Top-1 accuracy: 66.7%  (16/24 races)
  Top-3 accuracy: 95.8%  (23/24 races)


## Model 2: Linear Regression (the final shipped model)

Same 6 features, scaled, trained on 2022–2024. This should reproduce notebook 05's headline result.

In [5]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
y_pred_lr = lr_model.predict(X_test_scaled)

results_lr = evaluate_predictions(
    y_test, y_pred_lr, race_ids_test,
    label="Linear Regression (6 features)",
)


=== Linear Regression (6 features) ===
  RMSE:           4.221 positions
  MAE:            3.247 positions
  Top-1 accuracy: 58.3%  (14/24 races)
  Top-3 accuracy: 100.0%  (24/24 races)


/Users/ompatil9819gmail.com/F1-Race-Predictor/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/ompatil9819gmail.com/F1-Race-Predictor/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/ompatil9819gmail.com/F1-Race-Predictor/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_


## Model 3: Random Forest with light tuning

Random Forest doesn't need feature scaling. We tune `n_estimators`, `max_depth`, and `min_samples_split` via `TimeSeriesSplit` cross-validation on the training set — the same temporal-CV approach used for XGBoost in notebook 04.

In [6]:
# Sort training data by date so TimeSeriesSplit is meaningful
train_sorted = train_df.sort_values("EventDate").reset_index(drop=True)
X_train_sorted = train_sorted[features]
y_train_sorted = train_sorted["Position"]

tscv = TimeSeriesSplit(n_splits=3)

rf_param_grid = {
    "n_estimators": [100, 200, 400],
    "max_depth": [4, 6, 10, None],
    "min_samples_split": [2, 5, 10],
}

rf_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid=rf_param_grid,
    cv=tscv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    verbose=0,
)

print("Tuning Random Forest — 36 parameter combinations x 3 CV folds = 108 fits...")
print("(Takes around 30-60 seconds.)")
rf_search.fit(X_train_sorted, y_train_sorted)

print(f"\nBest CV RMSE: {-rf_search.best_score_:.3f}")
print(f"Best params:  {rf_search.best_params_}")

rf_model = rf_search.best_estimator_
y_pred_rf = rf_model.predict(X_test)

results_rf = evaluate_predictions(
    y_test, y_pred_rf, race_ids_test,
    label="Random Forest (tuned)",
)

Tuning Random Forest — 36 parameter combinations x 3 CV folds = 108 fits...
(Takes around 30-60 seconds.)

Best CV RMSE: 4.232
Best params:  {'max_depth': 4, 'min_samples_split': 10, 'n_estimators': 100}

=== Random Forest (tuned) ===
  RMSE:           4.209 positions
  MAE:            3.201 positions
  Top-1 accuracy: 45.8%  (11/24 races)
  Top-3 accuracy: 91.7%  (22/24 races)


## Model 4: Tuned XGBoost

Use the exact hyperparameters chosen in notebook 04 (`learning_rate=0.03, max_depth=3, min_child_weight=5, n_estimators=100`). Retrain on 2022–2024 (notebook 04 trained on 2022–2023), then predict 2025.

In [7]:
xgb_model = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.03,
    min_child_weight=5,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)

results_xgb = evaluate_predictions(
    y_test, y_pred_xgb, race_ids_test,
    label="XGBoost (tuned, notebook 04 hyperparameters)",
)


=== XGBoost (tuned, notebook 04 hyperparameters) ===
  RMSE:           4.239 positions
  MAE:            3.251 positions
  Top-1 accuracy: 45.8%  (11/24 races)
  Top-3 accuracy: 91.7%  (22/24 races)
